# Fine-tune Qwen2.5-Coder-7B Instruct with QLoRA — EduCode Rwanda v2

**Model:** `Qwen/Qwen2.5-Coder-7B-Instruct`  
**Dataset:** 2,301 train / 256 val (fixjs v2 + synthetic, all Mwarimu system prompt)  
**Environment:** Kaggle T4 GPU — single GPU only (`device_map={"":0}`). QLoRA with `device_map="auto"` causes a cross-device tensor error during loss computation; single-GPU avoids this entirely. 7B in 4-bit fits in one T4's 16 GB.  
**Stack:** `transformers` + `peft` (QLoRA) + `trl` (SFTTrainer) + `bitsandbytes`  
**Output:** `/kaggle/working/qwen-7b-educode-v2` → push to `DavBelaa/qwen25-coder-7b-educode-rwanda-v2`

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # single GPU — avoids cross-device QLoRA error
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
%%capture
!pip install -q -U bitsandbytes
!pip install -q transformers==4.46.3 peft==0.13.2 trl==0.12.1 accelerate==1.1.1 datasets==3.1.0 huggingface_hub

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {props.total_memory // 1024**3} GB")

## 1. Load and verify dataset

Upload `train_merged.jsonl` and `val_merged.jsonl` from `ml/data/processed/` to Kaggle as a dataset,
then update `DATA_DIR` below.

**Critical check:** every example must use the Mwarimu system prompt (not the old EduCode AI prompt).

In [ ]:
from datasets import load_dataset
from collections import Counter
import json

DATA_DIR = "/kaggle/input/educode-v2-training-data"

dataset = load_dataset(
    "json",
    data_files={
        "train":      f"{DATA_DIR}/train_merged.jsonl",
        "validation": f"{DATA_DIR}/val_merged.jsonl",
    },
)
print(dataset)

# ── System prompt verification ─────────────────────────────────────────────
all_examples = list(dataset["train"]) + list(dataset["validation"])
prompts = Counter(ex["messages"][0]["content"] for ex in all_examples)

print(f"\nUnique system prompts: {len(prompts)}  (must be 1)")
assert len(prompts) == 1, f"ERROR: found {len(prompts)} different system prompts!"

prompt_text = list(prompts.keys())[0]
assert prompt_text.startswith("You are Mwarimu"), "ERROR: system prompt does not start with 'You are Mwarimu'!"
assert "EduCode AI" not in prompt_text, "ERROR: old 'EduCode AI' prompt found!"
assert "Kinyarwanda translation is handled separately" in prompt_text, "ERROR: missing Kinyarwanda note!"

print("System prompt: OK — all examples use the Mwarimu prompt")
print(f"\nPrompt preview:\n{prompt_text[:200]}...")

## Dataset Visualizations

Understand the data before training: split sizes, source breakdown, and token length distribution.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from transformers import AutoTokenizer

# Load tokenizer here just for token counting (no GPU needed — tokenizer only)
_MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"
_tok = AutoTokenizer.from_pretrained(_MODEL_ID, trust_remote_code=True)

def _count_tokens(example):
    # dataset doesn't have 'text' yet (format_chat runs later) — join raw messages instead
    raw = " ".join(m["content"] for m in example["messages"])
    return len(_tok.encode(raw))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('EduCode Rwanda v2 — Dataset Overview', fontsize=14, fontweight='bold')

# ── 1. Train / Val split ──────────────────────────────────────────────────────
splits = ['Train', 'Validation']
sizes  = [len(dataset['train']), len(dataset['validation'])]
bars = axes[0].bar(splits, sizes, color=['#1565C0', '#E65100'], width=0.4)
axes[0].set_title('Train / Val Split')
axes[0].set_ylabel('Examples')
axes[0].set_ylim(0, max(sizes) * 1.15)
for bar, v in zip(bars, sizes):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 20, str(v),
                 ha='center', fontweight='bold', fontsize=11)

# ── 2. Source breakdown ───────────────────────────────────────────────────────
labels = ['fixjs v2\n(train)', 'Synthetic\n(train)', 'fixjs v2\n(val)', 'Synthetic\n(val)']
counts = [1401, 900, 156, 100]
colors = ['#1565C0', '#42A5F5', '#BF360C', '#FF8A65']
bars2  = axes[1].bar(labels, counts, color=colors)
axes[1].set_title('Source Breakdown')
axes[1].set_ylabel('Examples')
axes[1].set_ylim(0, max(counts) * 1.18)
for bar, v in zip(bars2, counts):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 10, str(v),
                 ha='center', fontweight='bold', fontsize=10)

# ── 3. Token length distribution (500-sample estimate) ────────────────────────
print("Counting token lengths (500 train sample + all val)...")
train_sample  = dataset['train'].select(range(500))
train_lengths = [_count_tokens(ex) for ex in train_sample]
val_lengths   = [_count_tokens(ex) for ex in dataset['validation']]

axes[2].hist(train_lengths, bins=30, alpha=0.7, color='#1565C0', label='Train (n=500 sample)')
axes[2].hist(val_lengths,   bins=30, alpha=0.7, color='#E65100', label=f'Val (n={len(val_lengths)})')
axes[2].axvline(1024, color='red', linestyle='--', linewidth=1.5, label='max_seq_length=1024')
axes[2].set_title('Token Length Distribution')
axes[2].set_xlabel('Tokens per example')
axes[2].set_ylabel('Count')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('/kaggle/working/dataset_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTrain — min={min(train_lengths)}  median={int(np.median(train_lengths))}  max={max(train_lengths)}")
print(f"Val   — min={min(val_lengths)}  median={int(np.median(val_lengths))}  max={max(val_lengths)}")
print(f"Examples over 1024 tokens (train sample): {sum(l > 1024 for l in train_lengths)}")
print(f"Examples over 1024 tokens (val):          {sum(l > 1024 for l in val_lengths)}")

## 2. Training time estimate

Print this before loading the model so we know we're within the 12-hour Kaggle cap.

In [ ]:
import math

N_TRAIN         = len(dataset["train"])
N_EPOCHS        = 3
BATCH_SIZE      = 1    # single GPU, batch_size=1 to stay within 16 GB
GRAD_ACCUM      = 16   # effective batch = 16
EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM

steps_per_epoch = math.ceil(N_TRAIN / EFFECTIVE_BATCH)
total_steps     = steps_per_epoch * N_EPOCHS

# v1 reference: 9h 42m = 34,920s for 2,250 steps on 3B single T4 (~15.5s/step)
# v2: 7B model on single T4, QLoRA — roughly 2-3× slower per step than 3B
SECS_PER_STEP_LOW  = 30
SECS_PER_STEP_HIGH = 50

est_low  = total_steps * SECS_PER_STEP_LOW  / 3600
est_high = total_steps * SECS_PER_STEP_HIGH / 3600

print("=" * 50)
print("TRAINING TIME ESTIMATE")
print("=" * 50)
print(f"Training examples:   {N_TRAIN:,}")
print(f"Epochs:              {N_EPOCHS}")
print(f"Effective batch:     {EFFECTIVE_BATCH}  (batch_size={BATCH_SIZE} × grad_accum={GRAD_ACCUM})")
print(f"Steps per epoch:     {steps_per_epoch}")
print(f"Total steps:         {total_steps}")
print(f"Estimated duration:  {est_low:.1f}h – {est_high:.1f}h")
print(f"Kaggle cap:          12h")
print(f"Safe within cap:     {'YES' if est_high < 11 else 'TIGHT — monitor closely'}")

## 3. Load Qwen2.5-Coder-7B in 4-bit (QLoRA)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4 is Turing (cc7.5) — no native bf16
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},                     # single GPU — fixes cross-device loss error
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {MODEL_ID}")
print(f"Total parameters: {total_params / 1e9:.2f}B")
print(f"Device: {next(model.parameters()).device}")

## 4. Apply LoRA adapters

Using `r=16` (up from `r=8` in v1) to give the 7B model more adapter capacity.

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Format with Qwen chat template

In [ ]:
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_chat, num_proc=2)

# Verify the Mwarimu name is in the formatted text
sample_text = dataset["train"][0]["text"]
assert "Mwarimu" in sample_text, "ERROR: Mwarimu not found in formatted training text!"
print("Chat template applied. Sample (first 300 chars):")
print(sample_text[:300])

## 6. Train

`eval_strategy="no"` — evaluation during training OOMs on T4 (7B logits × vocab 152k × fp32 ≈ 600 MB, no room left). Validation perplexity is computed separately in section 9 after training finishes.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
import time

OUTPUT_DIR = "/kaggle/working/qwen-7b-educode-v2"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=25,
    eval_strategy="no",
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=1024,
    packing=False,
)

start = time.time()
trainer.train()
elapsed = time.time() - start
print(f"\nTraining complete in {elapsed/3600:.2f}h ({elapsed/60:.0f} min)")

## 7. Training Loss Curve

Plot the loss from `trainer.state.log_history` (logged every 25 steps). A downward trend confirms the model learned from the data.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

train_logs = [x for x in trainer.state.log_history if 'loss' in x]

steps  = [x['step']  for x in train_logs]
losses = [x['loss']  for x in train_logs]
epochs = [x.get('epoch', 0) for x in train_logs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Training Dynamics — Qwen2.5-Coder-7B EduCode Rwanda v2', fontsize=13, fontweight='bold')

# Loss vs steps (raw + smoothed)
ax1.plot(steps, losses, color='#90CAF9', linewidth=1, alpha=0.7, label='Step loss')
if len(losses) >= 5:
    smooth = np.convolve(losses, np.ones(5)/5, mode='valid')
    ax1.plot(steps[4:], smooth, color='#1565C0', linewidth=2, label='Smoothed (5-step avg)')
ax1.set_xlabel('Training Step')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Loss vs Step')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss vs epoch
ax2.plot(epochs, losses, color='#A5D6A7', linewidth=1, alpha=0.7, label='Step loss')
if len(losses) >= 5:
    ax2.plot(epochs[4:], smooth, color='#2E7D32', linewidth=2, label='Smoothed')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Cross-Entropy Loss')
ax2.set_title('Loss vs Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/training_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Total steps logged: {len(steps)}")
print(f"Initial loss:       {losses[0]:.4f}")
print(f"Final loss:         {losses[-1]:.4f}")
print(f"Reduction:          {(losses[0]-losses[-1])/losses[0]*100:.1f}%")

In [ ]:
from huggingface_hub import login

# Read HF_TOKEN from Kaggle Secrets.
# In Kaggle: notebook Settings → Secrets → Add New Secret (Key: HF_TOKEN, Value: your token)
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found. Add it in Kaggle Settings → Secrets.")

login(token=HF_TOKEN)

HF_REPO = "DavBelaa/qwen25-coder-7b-educode-rwanda-v2"

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")

model.push_to_hub(HF_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
print(f"\nPushed to: https://huggingface.co/{HF_REPO}")

## 8. Quick sanity check — sample inference

## 9. Evaluation — Perplexity & Qualitative Examples

**Perplexity** is the standard metric for language models. It measures how "surprised" the model is by the validation set — lower is better. We compute it with `max_length=512` to avoid the same OOM that blocked in-training eval.

**Qualitative examples** show the model responding to real student scenarios — the most compelling evidence for a defense panel.

In [ ]:
import math

# ── Perplexity ────────────────────────────────────────────────────────────────
# max_length=512 (not 1024) to avoid OOM — no gradient checkpointing during eval
EVAL_SAMPLES = 80
EVAL_MAX_LEN = 512

print(f"Computing perplexity on {EVAL_SAMPLES} validation examples (max_length={EVAL_MAX_LEN})...")
model.eval()
total_loss, n = 0.0, 0

with torch.no_grad():
    for example in dataset['validation'].select(range(EVAL_SAMPLES)):
        inputs = tokenizer(
            example['text'], return_tensors='pt',
            truncation=True, max_length=EVAL_MAX_LEN,
        ).to('cuda:0')
        if inputs['input_ids'].shape[1] < 10:
            continue
        outputs = model(**inputs, labels=inputs['input_ids'])
        total_loss += outputs.loss.item()
        n += 1

avg_loss   = total_loss / n
perplexity = math.exp(avg_loss)

print(f"\n{'='*50}")
print(f"  Avg validation loss:   {avg_loss:.4f}")
print(f"  Validation perplexity: {perplexity:.2f}")
print(f"{'='*50}")
print(f"\n  A random model over Qwen's 152,064-token vocab would score ~152,000.")
print(f"  A well-fine-tuned model on this domain typically scores < 5.")

# ── Qualitative examples ──────────────────────────────────────────────────────
print("\n\n" + "="*65)
print("QUALITATIVE EVALUATION — sample model responses")
print("="*65)

TEST_CASES = [
    {
        "label": "const reassignment",
        "message": (
            "This code has an error. Explain what is wrong and how to fix it:\n"
            "```javascript\nconst score = 0;\nscore = score + 10;\nconsole.log(score);\n```\n"
            "Error: TypeError: Assignment to constant variable."
        ),
    },
    {
        "label": "undefined variable (TDZ)",
        "message": (
            "This code has an error. Explain what is wrong and how to fix it:\n"
            "```javascript\nconsole.log(name);\nlet name = 'Alice';\n```\n"
            "Error: ReferenceError: Cannot access 'name' before initialization"
        ),
    },
    {
        "label": "missing return value",
        "message": (
            "This code has an error. Explain what is wrong and how to fix it:\n"
            "```javascript\nfunction add(a, b) {\n    let result = a + b;\n}\nconsole.log(add(3, 4));\n```\n"
            "Output is: undefined"
        ),
    },
    {
        "label": "concept question — arrow functions",
        "message": "What is the difference between a regular function and an arrow function in JavaScript?",
    },
]

for tc in TEST_CASES:
    messages   = [{"role": "user", "content": tc["message"]}]
    prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs     = tokenizer(prompt_str, return_tensors="pt", truncation=True, max_length=EVAL_MAX_LEN).to("cuda:0")

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=180,
            temperature=0.3,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    print(f"\n{'─'*65}")
    print(f"SCENARIO: {tc['label']}")
    print(f"INPUT:    {tc['message'][:120]}{'...' if len(tc['message'])>120 else ''}")
    print(f"\nMWARIMU:\n{response}")

In [ ]:
model.eval()

TEST_PROMPT = "What is wrong with this JavaScript code?\n\n```javascript\nconst x = 5;\nx = 10;\nconsole.log(x);\n```"

messages = [{"role": "user", "content": TEST_PROMPT}]
prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda:0")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.3,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("[USER]")
print(TEST_PROMPT)
print("\n[MWARIMU]")
print(response)